In [1]:

from pathlib import Path

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

ROOT = Path(r"D:\PS07_CT2MRI")

PATIENTS = ["P2658", "P6123", "P7363", "P7480", "P8947"]

# Select the central slice initially
OUTPUT_DIR = ROOT / "results" / "model_comparison"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def load_volume(path):
    return np.asarray(
        nib.load(str(path)).dataobj,
        dtype=np.float32
    )


for patient in PATIENTS:

    folder = ROOT / "dataset" / "volumes" / patient

    ct = load_volume(folder / "ct.nii.gz")
    mr = load_volume(folder / "mr.nii.gz")
    mask = load_volume(folder / "mask.nii.gz")

    model1 = load_volume(
        ROOT / "outputs" / "validation" /
        f"{patient}_predicted_mri.nii.gz"
    )

    model2 = load_volume(
        ROOT / "outputs" / "pix2pix_best_epoch5" /
        f"{patient}_predicted_mri.nii.gz"
    )

    if not (ct.shape == mr.shape == model1.shape == model2.shape):
        raise ValueError(f"Volume shape mismatch: {patient}")

    # Choose the middle slice containing the supplied mask
    valid_slices = [
        z for z in range(mask.shape[2])
        if np.any(mask[:, :, z] > 0)
    ]

    z = valid_slices[len(valid_slices) // 2]

    fig, axes = plt.subplots(1, 5, figsize=(20, 5))

    images = [
        (ct[:, :, z], "CT Input"),
        (mr[:, :, z], "Actual MRI"),
        (model1[:, :, z], "Residual U-Net"),
        (model2[:, :, z], "Pix2Pix Epoch 5"),
        (np.abs(model1[:, :, z] - model2[:, :, z]),
         "Absolute Difference")
    ]

    for ax, (image, title) in zip(axes, images):

        ax.imshow(
            np.rot90(image),
            cmap="gray",
            vmin=0,
            vmax=1
        )

        ax.set_title(title)
        ax.axis("off")

    fig.suptitle(
        f"{patient} | Slice {z}",
        fontsize=15
    )

    plt.tight_layout()

    save_path = OUTPUT_DIR / f"{patient}_comparison.png"

    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

    print("Saved:", save_path)

print("\nAll comparison images generated.")


Saved: D:\PS07_CT2MRI\results\model_comparison\P2658_comparison.png
Saved: D:\PS07_CT2MRI\results\model_comparison\P6123_comparison.png
Saved: D:\PS07_CT2MRI\results\model_comparison\P7363_comparison.png
Saved: D:\PS07_CT2MRI\results\model_comparison\P7480_comparison.png
Saved: D:\PS07_CT2MRI\results\model_comparison\P8947_comparison.png

All comparison images generated.
